# 결제 지연 위험 (10회차)

식 (10.1) $\mathbb{E}[\text{손실}](L)=(1-e^{-hL})\,N\sigma\sqrt L/\sqrt{2\pi}$를 Lab C 코드로 다시 계산하고, 몬테카를로·탄력성·두꺼운 꼬리를 그림으로 본다. 가정은 Lab C와 같다: $N=10$억 원, $\sigma=1\%$/일, $h=0.10/250$.

In [ ]:
import sys, pathlib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
LABS = pathlib.Path.cwd().parent / "labs"
for f in ["Noto Sans CJK KR", "AppleGothic", "Malgun Gothic", "NanumGothic"]:
    if any(f == x.name for x in font_manager.fontManager.ttflist):
        plt.rcParams["font.family"] = f; break
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110,
                     "axes.unicode_minus": False})
ACC, GREY = "#2F5D8C", "#8E8C87"
sys.path.insert(0, str(LABS / 'lab-c-settlement'))
import settlement_risk as lc
print(lc.SIGMA, lc.H, lc.NOTIONAL)

## 1. 닫힌 해와 몬테카를로

`closed_form`은 식 (10.1)을 그대로 옮긴 함수다. 같은 가정으로 부도 여부와 가격 변동을 직접 뽑아 평균을 내면(시드 고정) 같은 값 근처가 나와야 한다.

In [ ]:
rng = np.random.default_rng(20261028)
rows = []
for L in [2, 1, 0.5, 1/24]:
    cf = lc.closed_form(L) * lc.NOTIONAL
    mc, se = lc.monte_carlo(L, rng)
    rows.append((L, cf, mc * lc.NOTIONAL, se * lc.NOTIONAL))
    print(f"L={L:6.3f}일  닫힌 해 {cf:9.1f}원   MC {mc*lc.NOTIONAL:9.1f}원 (표준오차 {se*lc.NOTIONAL:.1f})")

## 2. 곡선: $L^{1.5}$

로그 축에서 기울기 1.5인 직선이 된다. T+2 → T+1이 $2^{1.5}\approx2.83$배인 이유다.

In [ ]:
L = np.logspace(np.log10(1/48), np.log10(5), 200)
E = np.array([lc.closed_form(x) for x in L]) * lc.NOTIONAL
fig, ax = plt.subplots(figsize=(6.4, 3.6))
ax.loglog(L, E, color=ACC)
for x, lab in [(2, "T+2"), (1, "T+1"), (0.5, "반나절"), (1/24, "1시간")]:
    y = lc.closed_form(x) * lc.NOTIONAL
    ax.plot(x, y, "o", color=ACC); ax.annotate(f"{lab}\n{y:,.1f}원", (x, y), textcoords="offset points", xytext=(6, -14), fontsize=8)
ax.set_xlabel("결제 지연 L (영업일, 로그)"); ax.set_ylabel("기대 대체비용 (원, 로그)")
ax.set_title("10억 원 한 건의 기대 대체비용", fontsize=10); plt.tight_layout()

## 3. 탄력성 $\varepsilon = x/(e^x-1)+1/2$

$x=hL$이 작으면 1.5, 부도 확률이 커지면 0.5 쪽으로 내려간다(문제집 10.2).

In [ ]:
x = np.logspace(-4, 1.3, 200)
eps = x / np.expm1(x) + 0.5
fig, ax = plt.subplots(figsize=(6.4, 3.2))
ax.semilogx(x, eps, color=ACC); ax.axhline(1.5, color=GREY, lw=.6, ls="--"); ax.axhline(0.5, color=GREY, lw=.6, ls="--")
ax.plot(lc.H, 0.5 + lc.H/np.expm1(lc.H), "o", color=ACC); ax.annotate("Lab C 가정 (x=0.0004)", (lc.H, 1.5), xytext=(4, -16), textcoords="offset points", fontsize=8)
ax.set_xlabel("x = hL"); ax.set_ylabel("탄력성 ε"); plt.tight_layout()
print("x=1일 때 ε =", round(1/np.expm1(1) + 0.5, 3))

## 4. 꼬리가 두꺼우면

분산을 1로 맞춘 $t_3$와 정규분포. 양의 부분의 평균은 $t_3$가 더 작지만(0.318 대 0.399), 99% 분위수는 더 크다(2.62 대 2.33).

In [ ]:
rng = np.random.default_rng(7)
z = rng.standard_normal(2_000_000)
t = rng.standard_t(3, 2_000_000) / np.sqrt(3)
for name, s in [("정규", z), ("t3 (분산 1)", t)]:
    print(f"{name:10s}  E[X+] = {np.maximum(s, 0).mean():.3f}   99% 분위수 = {np.quantile(s, .99):.2f}")
print("이론값: 1/sqrt(2π) =", round(1/np.sqrt(2*np.pi), 3), ", 1/π =", round(1/np.pi, 3), ", 4.541/sqrt(3) =", round(4.541/np.sqrt(3), 2))

**읽을 것.** 기대손실만 보면 꼬리가 두꺼운 상품이 더 안전해 보인다. 결제 방식을 고를 때는 분위수(한 번의 큰 손실)를 함께 봐야 한다.

출처: Lab C `settlement_risk.py`; Duffie and Singleton (2003) *Credit Risk*; Mandelbrot (1963).